In [2]:
import random
import numpy as np

# Envirenment

In [28]:
NUM_STATES = 5                           # Number of states
NUM_ACTIONS = 2
TERMINAL_STATE = 4                          
GAMMA = 0.9

START_STATE = 0
GOAL_STATE = 4

LEFT = 0
RIGHT = 1


In [4]:
def step(state, action):
    """
    Applies an action and returns: next_state, reward, terminated
    """
    if action == 0:
        state -= 1
        if state < 0 : state = 0
    elif action == 1:
        state += 1
        if state > GOAL_STATE: state = GOAL_STATE
    
    if state == GOAL_STATE:
        reward = 1
    else:
        reward = 0
    
    if state == GOAL_STATE: terminated = True
    else: terminated = False
    
    return state, reward, terminated

# Policy

In [49]:
def choose_action():
    """
    Delivers next action based on Random policy: choose LEFT or RIGHT with equal probability of 50%.
    """
    return random.randint(0, 1)

# Episode Generation

In [63]:
def run_episode(max_steps=100, fixed_action_policy=None):
    """
    Runs one episode and return its trajectory.
    Each trajectory element should be: (state, action, reward)
    optional fixed_action_policy epecifically for evaluating against random action policy. 
    """
    state = START_STATE
    trajectory = []
    for _ in range(max_steps):
        if fixed_action_policy==None:
            action = choose_action()
        else:
            action = fixed_action_policy
        next_state, reward, termination = step(state=state, action=action)
        trajectory.append((state, action, reward))
        state = next_state
        if termination: break
    
    return trajectory

# Return calculation

In [7]:
def compute_returns(trajectory, gamma=GAMMA):
    """
    Compute discounted return G_t for every transition in the trajectory.
    """
    reversed_returns = []
    for i, val in reversed(list(enumerate(trajectory))):    # Enumerating backward
        reward = val[2]
        if reversed_returns:
            G = reward + gamma * reversed_returns[-1]
            reversed_returns.append(G)
        else:
            reversed_returns.append(reward)
    return list(reversed(reversed_returns))

# Update Q(s, a)

In [10]:
def update_q_from_episode(
        trajectory, 
        Q, 
        visit_counts, 
        gamma=GAMMA
):
    """
    Every-visit Monte Carlo update for Q(s, a).
    """
    returns = compute_returns(trajectory, gamma)

    for transition, ret in zip(trajectory, returns):
        state =  transition[0]
        action = transition[1]
        G_t = ret
        visit_counts[state, action] += 1
        Q[state, action] += (G_t - Q[state, action]) / visit_counts[state, action]

    return Q, visit_counts

# Training

In [29]:
def train_mc_q(num_episodes, gamma=GAMMA):
    # initialize Q from zero then train
    Q = np.zeros((NUM_STATES, NUM_ACTIONS), dtype=float)
    visit_counts = np.zeros((NUM_STATES, NUM_ACTIONS), dtype=int)

    for episode in range(num_episodes):
        trajectory = run_episode()
        Q, visit_counts = update_q_from_episode(trajectory, Q, visit_counts, gamma)
    
    return Q, visit_counts

In [22]:
def greedy_action(state, Q):
    """
    Return the action with the largest Q-value for the given state.
    """
    return np.argmax(Q[state])

# Runing

In [41]:
ep = [1, 10, 100, 1000]
res = {}
for e in ep:
    Q, visit_counts = train_mc_q(e)
    greedy_actions = {}
    for s in range(NUM_STATES - 1):            # "NUM_STATES - 1": Excluding the terminal state. 
        greedy = greedy_action(s, Q)
        greedy_actions[s] = {"greedy_action of this state": greedy}
    res[e] = {"Q": Q, "visit_counts": visit_counts, "greedy actions": greedy_actions}

In [42]:
print('Results for 1 episode:')
display(res[1])

Results for 1 episode:


{'Q': array([[0.4226258 , 0.46958422],
        [0.4782969 , 0.52176024],
        [0.43046721, 0.8145    ],
        [0.81      , 1.        ],
        [0.        , 0.        ]]),
 'visit_counts': array([[2, 2],
        [1, 2],
        [1, 2],
        [1, 1],
        [0, 0]]),
 'greedy actions': {0: {'greedy_action of this state': np.int64(1)},
  1: {'greedy_action of this state': np.int64(1)},
  2: {'greedy_action of this state': np.int64(1)},
  3: {'greedy_action of this state': np.int64(1)}}}

In [43]:
print('Results for 10 episode:')
display(res[10])

Results for 10 episode:


{'Q': array([[0.18381656, 0.19142864],
        [0.12404581, 0.29615106],
        [0.18766482, 0.46561106],
        [0.19557603, 1.        ],
        [0.        , 0.        ]]),
 'visit_counts': array([[63, 55],
        [45, 46],
        [36, 25],
        [15, 10],
        [ 0,  0]]),
 'greedy actions': {0: {'greedy_action of this state': np.int64(1)},
  1: {'greedy_action of this state': np.int64(1)},
  2: {'greedy_action of this state': np.int64(1)},
  3: {'greedy_action of this state': np.int64(1)}}}

In [44]:
print('Results for 100 episode:')
display(res[100])

Results for 100 episode:


{'Q': array([[0.24459335, 0.30632289],
        [0.2574515 , 0.44546178],
        [0.33568409, 0.63072075],
        [0.4360228 , 1.        ],
        [0.        , 0.        ]]),
 'visit_counts': array([[401, 392],
        [292, 289],
        [189, 213],
        [113, 100],
        [  0,   0]]),
 'greedy actions': {0: {'greedy_action of this state': np.int64(1)},
  1: {'greedy_action of this state': np.int64(1)},
  2: {'greedy_action of this state': np.int64(1)},
  3: {'greedy_action of this state': np.int64(1)}}}

In [45]:
print('Results for 1000 episode:')
display(res[1000])

Results for 1000 episode:


{'Q': array([[0.23483719, 0.28994055],
        [0.23391606, 0.41899162],
        [0.29966323, 0.63489597],
        [0.41961469, 1.        ],
        [0.        , 0.        ]]),
 'visit_counts': array([[4087, 4100],
        [3101, 3059],
        [2061, 2023],
        [1025,  997],
        [   0,    0]]),
 'greedy actions': {0: {'greedy_action of this state': np.int64(1)},
  1: {'greedy_action of this state': np.int64(1)},
  2: {'greedy_action of this state': np.int64(1)},
  3: {'greedy_action of this state': np.int64(1)}}}

# Evaluation: Greedy against Random action policy
Based on previous results, we know that the greedy action is 1.

In [82]:
def evaluation(res, num_episodes):
    """
    Evaluation based on average episode length.
    """
    average_lengtth_random_policy = sum(sum(res[num_episodes]['visit_counts'])) / num_episodes
    greedy_action = res[num_episodes]['greedy actions'][0]['greedy_action of this state']

    greedy_length = []
    for e in range(num_episodes):
        trajectory = run_episode(fixed_action_policy=greedy_action)
        greedy_length.append(len(trajectory))
    
    average_greedy_length = np.mean(greedy_length)

    return f"the average episode length over {num_episodes} episodes using random policy is: {average_lengtth_random_policy}, against: {average_greedy_length} by using greedy policy of action=1!"
    

In [83]:
evaluation(res, 1000)

'the average episode length over 1000 episodes using random policy is: 20.453, against: 4.0 by using greedy policy of action=1!'